# Day 5 - Tokens, sampling, and what 1,000 alerts cost

**Big idea:** models don't read words, they read **tokens** - and you pay per token. Know how many you use and you know what you'll pay.

## 1. What a token looks like

A tokenizer chops text into chunks from its vocabulary. Here are the actual chunks, separated by `|`:

In [1]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")   # -> o200k_base
print("encoding:", enc.name)

def show_tokens(text, enc=enc):
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{len(ids):>2} tokens: " + "|".join(pieces))

show_tokens("Machine M-14 temperature 92C exceeds threshold 85C, tenant=acme")
show_tokens("hello")
show_tokens("antidisestablishmentarianism")

encoding: o200k_base
18 tokens: Machine| M|-|14| temperature| |92|C| exceeds| threshold| |85|C|,| tenant|=|ac|me
 1 tokens: hello
 6 tokens: ant|idis|est|ablishment|arian|ism


Common words are one token; rare words break into several pieces. Rule of thumb for English: **~4 characters per token**.

## 2. How tokenizers are built: BPE in 30 seconds

1. Start with raw bytes (256 possible symbols).
2. Find the pair of symbols that shows up together most often in the training text. Merge it into one new symbol.
3. Repeat thousands of times -> a vocabulary of ~100k-200k tokens.

Each company trains on **different text**, so each ends up with a **different vocabulary**. Same sentence, different ruler, different count:

In [2]:
old = tiktoken.get_encoding("cl100k_base")   # GPT-4 / GPT-3.5
new = tiktoken.get_encoding("o200k_base")    # GPT-4o

samples = [
    "Machine M-14 temperature 92C exceeds threshold 85C",
    "機械が過熱しています",                     # Japanese: "the machine is overheating"
    "def two_sum(nums, target): return []",
]
for s in samples:
    print(f"cl100k={len(old.encode(s)):>3}   o200k={len(new.encode(s)):>3}   {s}")

cl100k= 13   o200k= 13   Machine M-14 temperature 92C exceeds threshold 85C
cl100k= 14   o200k=  6   機械が過熱しています
cl100k=  9   o200k=  9   def two_sum(nums, target): return []


Non-English text is where vocabularies disagree most. And for Claude, tiktoken has **no** encoding at all - any tiktoken count for Claude is an **estimate** and must be labeled as one. (Exact counts need Anthropic's own `count_tokens` endpoint.)

## 3. Context window: the model's desk size

The **context window** is the most tokens a model can look at at once (prompt + answer). Go over it and the API rejects the request - or something upstream silently cuts text off, which is worse because nothing visibly breaks.

So check the length *before* sending:

In [3]:
def fits(text, limit_tokens, enc=enc):
    n = len(enc.encode(text))
    return n <= limit_tokens, n

huge_alert = "Machine M-14 overheating. " * 2000   # ~50,000 characters
ok, n = fits(huge_alert, limit_tokens=8_000)
print(f"{len(huge_alert):,} chars -> {n:,} tokens -> fits in an 8k window? {ok}")

52,000 chars -> 12,001 tokens -> fits in an 8k window? False


## 4. Sampling: temperature and top-p

After reading your prompt, the model gives a **probability for every possible next token**. Then it *picks* one:

- **Temperature** reshapes those probabilities. Low (near 0) = almost always the top choice. High = spread out, so surprises win more often.
- **Top-p** keeps only the smallest set of top tokens whose probabilities add up to p (e.g. 0.9), then picks from those. It cuts off the weird tail.

Toy example - the model is choosing the severity word for an alert:

In [4]:
import math

words  = ["critical", "high", "medium", "banana"]
logits = [3.0, 2.0, 1.0, -1.0]      # raw scores from the model

def softmax(scores, temperature):
    scaled = [s / temperature for s in scores]
    m = max(scaled)
    exps = [math.exp(s - m) for s in scaled]
    total = sum(exps)
    return [e / total for e in exps]

for t in (0.2, 1.0, 2.0):
    probs = softmax(logits, t)
    row = "  ".join(f"{w}={p:.2f}" for w, p in zip(words, probs))
    print(f"T={t}:  {row}")

T=0.2:  critical=0.99  high=0.01  medium=0.00  banana=0.00
T=1.0:  critical=0.66  high=0.24  medium=0.09  banana=0.01
T=2.0:  critical=0.47  high=0.29  medium=0.17  banana=0.06


In [5]:
def top_p_filter(words, probs, p=0.9):
    ranked = sorted(zip(words, probs), key=lambda x: x[1], reverse=True)
    kept, total = [], 0.0
    for w, pr in ranked:
        kept.append(w)
        total += pr
        if total >= p:
            break
    return kept

probs = softmax(logits, 1.0)
print("top-p 0.9 keeps:", top_p_filter(words, probs, 0.9))
print("top-p 0.5 keeps:", top_p_filter(words, probs, 0.5))

top-p 0.9 keeps: ['critical', 'high', 'medium']
top-p 0.5 keeps: ['critical']


At T=0.2, "critical" wins almost every time. At T=2.0 even "banana" gets a real chance. For alert triage use **low temperature**: the same alert should get the same boring, consistent summary every time.

## 5. What 1,000 alerts cost

This reuses the real Day 5 code: the 20 synthetic alerts and the prices I looked up from each provider's own pricing page. They're **imported**, not typed again, so there's one source of truth.

In [6]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

import sys

sys.path.insert(0, str(ROOT / "days" / "day-005"))
from alerts import ALERTS
from token_cost import MODELS, OUTPUT_TOKENS_PER_ALERT

for name, cfg in MODELS.items():
    counts = [len(cfg["encoding"].encode(a)) for a in ALERTS]
    avg_in = sum(counts) / len(counts)
    price_in = cfg["input_price_per_1m"] / 1_000_000     # $ per single token
    price_out = cfg["output_price_per_1m"] / 1_000_000
    per_1000 = (avg_in * price_in + OUTPUT_TOKENS_PER_ALERT * price_out) * 1000
    label = "estimate" if cfg["approx"] else "exact"
    print(f"{name:<16} ({label:<8}) avg {avg_in:.1f} in + {OUTPUT_TOKENS_PER_ALERT} out tokens -> ${per_1000:.4f} per 1,000 alerts")

gpt-4o           (exact   ) avg 26.1 in + 40 out tokens -> $0.4653 per 1,000 alerts
claude-sonnet-5  (estimate) avg 26.1 in + 40 out tokens -> $0.4522 per 1,000 alerts


Less than half a dollar per 1,000 alerts. Two honest caveats:
- This multiplies an **average** by 1,000. Real bills **add up each request's actual tokens**. Fine here (every alert is 21-29 tokens), misleading when lengths vary a lot.
- The claude-sonnet-5 line uses a GPT tokenizer as a stand-in, so it's an **estimate**.

## 6. Why output tokens cost more than input

- **Input:** the whole prompt is processed **in one parallel pass**.
- **Output:** the model generates **one token at a time**, each needing its own pass.

More compute per token -> higher price. For gpt-4o: $2.50 in vs $10.00 out per 1M (4x).

## Recap

- A token is about 4 English characters; every model chunks text differently (BPE trained on different data).
- tiktoken is exact only for OpenAI models; label everything else as an estimate.
- Check length against the context window *before* sending.
- Low temperature = consistent output. Top-p trims the unlikely tail.
- Cost = tokens x price, input and output separately. Output is pricier.